## NGIF-03

# espírito do NGIF-03: não forçar o dataset a confirmar nossa hipótese, mas usar os dados para descobrir quais perguntas merecem ser investigadas.

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


Carregando e eexplorando a base 

In [27]:
import pandas as pd
import numpy as np

caminho = (
    'C:/Users/SAMSUNG/Desktop/'
    'deteccao de fraudes financeiras/'
    'PS_20174392719_1491204439457_log.csv.zip'
)

df = pd.read_csv(caminho, compression='zip')

print("Arquivo carregado com sucesso!")
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

Arquivo carregado com sucesso!
Linhas: 6,362,620
Colunas: 11


conhecendo a extrutura

In [28]:
display(df.head())

print("\nTipos das colunas:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isna().sum())

print("\nLinhas duplicadas:")
print(df.duplicated().sum())

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0



Tipos das colunas:
step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Valores ausentes por coluna:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Linhas duplicadas:
0


começar a investigação

In [29]:
print("Tipos de transação:")
display(
    df["type"]
    .value_counts()
    .rename_axis("tipo")
    .reset_index(name="quantidade")
)

print("\nDistribuição de isFraud:")
display(
    df["isFraud"]
    .value_counts()
    .rename_axis("isFraud")
    .reset_index(name="quantidade")
)

print("\nPercentual de cada classe:")
display(
    (df["isFraud"].value_counts(normalize=True) * 100)
    .rename_axis("isFraud")
    .reset_index(name="percentual")
    .round(4)
)

Tipos de transação:


,tipo,quantidade
0,CASH_OUT,2237500
1,PAYMENT,2151495
2,CASH_IN,1399284
3,TRANSFER,532909
4,DEBIT,41432



Distribuição de isFraud:


,isFraud,quantidade
0,0,6354407
1,1,8213



Percentual de cada classe:


,isFraud,percentual
0,0,99.8709
1,1,0.1291


entender o comportamento financeiro

In [30]:
print("Estatísticas dos valores:")
display(df["amount"].describe())

print("\nValores por tipo de transação:")
display(
    df.groupby("type")["amount"]
    .agg(["count", "mean", "median", "max"])
    .round(2)
)

print("\nFraudes por tipo de transação:")
display(
    pd.crosstab(
        df["type"],
        df["isFraud"],
        margins=True
    )
)

Estatísticas dos valores:


count    6.362620e+06
mean     1.798619e+05
std      6.038582e+05
min      0.000000e+00
25%      1.338957e+04
50%      7.487194e+04
75%      2.087215e+05
max      9.244552e+07
Name: amount, dtype: float64


Valores por tipo de transação:


,count,mean,median,max
type,,,,
CASH_IN,1399284,168920.24,143427.71,1915267.90
CASH_OUT,2237500,176273.96,147072.18,10000000.00
DEBIT,41432,5483.67,3048.99,569077.51
PAYMENT,2151495,13057.60,9482.19,238637.98
TRANSFER,532909,910647.01,486308.39,92445516.64



Fraudes por tipo de transação:


isFraud,0,1,All
type,,,
CASH_IN,1399284,0,1399284
CASH_OUT,2233384,4116,2237500
DEBIT,41432,0,41432
PAYMENT,2151495,0,2151495
TRANSFER,528812,4097,532909
All,6354407,8213,6362620


Investigação preliminar das transações rotuladas como fraude

In [31]:
fraudes = df.loc[df["isFraud"] == 1].copy()

print(f"Transações rotuladas como fraude: {len(fraudes):,}")
print(f"Origens distintas: {fraudes['nameOrig'].nunique():,}")
print(f"Destinos distintos: {fraudes['nameDest'].nunique():,}")

print("\nDistribuição por tipo:")
display(
    fraudes["type"].value_counts().to_frame("quantidade")
)

print("\nEstatísticas dos valores das transações rotuladas:")
display(
    fraudes.groupby("type")["amount"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

Transações rotuladas como fraude: 8,213
Origens distintas: 8,213
Destinos distintos: 8,169

Distribuição por tipo:


,quantidade
type,
CASH_OUT,4116
TRANSFER,4097



Estatísticas dos valores das transações rotuladas:


,count,mean,median,min,max
type,,,,,
CASH_OUT,4116,1455102.59,435516.90,0.0,10000000.0
TRANSFER,4097,1480891.67,445705.76,63.8,10000000.0


Contas que aparecem repetidamente como origem

In [32]:
origens = (
    fraudes.groupby("nameOrig")
    .agg(
        transacoes=("nameOrig", "size"),
        valor_total=("amount", "sum"),
        tipos=("type", "nunique"),
        primeiro_step=("step", "min"),
        ultimo_step=("step", "max")
    )
    .sort_values("transacoes", ascending=False)
)

print("Origens com mais transações rotuladas como fraude:")
display(origens.head(15))

# Contas que aparecem repetidamente como destino
destinos = (
    fraudes.groupby("nameDest")
    .agg(
        transacoes=("nameDest", "size"),
        valor_total=("amount", "sum"),
        tipos=("type", "nunique"),
        primeiro_step=("step", "min"),
        ultimo_step=("step", "max")
    )
    .sort_values("transacoes", ascending=False)
)

print("Destinos com mais transações rotuladas como fraude:")
display(destinos.head(15))

Origens com mais transações rotuladas como fraude:


,transacoes,valor_total,tipos,primeiro_step,ultimo_step
nameOrig,,,,,
C1000036340,1,253648.68,1,655,655
C1000086512,1,33676.59,1,95,95
C1000331499,1,2016790.84,1,262,262
C1000484178,1,3018810.85,1,504,504
C1000513158,1,40388.58,1,330,330
C1000937208,1,1534985.07,1,82,82
C1001502110,1,484493.06,1,357,357
C1001765380,1,277048.48,1,608,608
C1001875185,1,3288377.47,1,674,674


Destinos com mais transações rotuladas como fraude:


,transacoes,valor_total,tipos,primeiro_step,ultimo_step
nameDest,,,,,
C1399829166,2,3540009.28,1,212,314
C14138104,2,4597314.28,2,415,436
C1656842502,2,524000.74,2,154,537
C1497532505,2,744832.46,1,195,306
C1650668671,2,46972.83,1,201,568
C1325541393,2,350800.27,1,141,387
C104038589,2,2202769.97,2,39,59
C1780714769,2,717835.16,2,164,600
C185805228,2,1124385.07,1,16,18


 Concentração de transações rotuladas como fraude

In [33]:
freq_destinos = (
    fraudes["nameDest"]
    .value_counts()
    .rename_axis("conta_destino")
    .reset_index(name="qtd_fraudes")
)

print("Distribuição de destinos por quantidade de transações:")
display(freq_destinos["qtd_fraudes"].value_counts().sort_index())

print(
    "\nDestinos que recebem mais de uma transação rotulada:",
    (freq_destinos["qtd_fraudes"] > 1).sum()
)

print(
    "Maior quantidade de transações por destino:",
    freq_destinos["qtd_fraudes"].max()
)

Distribuição de destinos por quantidade de transações:


qtd_fraudes
1    8125
2      44
Name: count, dtype: int64


Destinos que recebem mais de uma transação rotulada: 44
Maior quantidade de transações por destino: 2


Hipótese de transferência seguida de saque

In [34]:
transfers = (
    fraudes.loc[fraudes["type"] == "TRANSFER",
                ["step", "nameOrig", "nameDest", "amount"]]
    .rename(columns={
        "step": "step_transfer",
        "nameOrig": "origem_transfer",
        "nameDest": "conta_intermediaria",
        "amount": "valor_transfer"
    })
)

cashouts = (
    fraudes.loc[fraudes["type"] == "CASH_OUT",
                ["step", "nameOrig", "nameDest", "amount"]]
    .rename(columns={
        "step": "step_cashout",
        "nameOrig": "origem_cashout",
        "nameDest": "destino_cashout",
        "amount": "valor_cashout"
    })
)

candidatas = transfers.merge(
    cashouts,
    left_on="conta_intermediaria",
    right_on="origem_cashout",
    how="inner"
)

# Manter somente saques ocorridos em um step posterior
sequencias = candidatas.loc[
    candidatas["step_cashout"] > candidatas["step_transfer"]
].copy()

sequencias["intervalo_steps"] = (
    sequencias["step_cashout"] - sequencias["step_transfer"]
)

print("Conexões encontradas, antes do filtro temporal:",
      len(candidatas))

print("Sequências com saque em step posterior:",
      len(sequencias))

display(
    sequencias.sort_values(
        ["intervalo_steps", "valor_transfer"],
        ascending=[True, False]
    ).head(20)
)

Conexões encontradas, antes do filtro temporal: 0
Sequências com saque em step posterior: 0


,step_transfer,origem_transfer,conta_intermediaria,valor_transfer,step_cashout,origem_cashout,destino_cashout,valor_cashout,intervalo_steps


 Investigação temporal em toda a base

In [35]:
transfers_all = (
    df.loc[df["type"] == "TRANSFER",
           ["step", "nameOrig", "nameDest", "amount", "isFraud"]]
    .rename(columns={
        "step": "step_transfer",
        "nameOrig": "origem_transfer",
        "nameDest": "conta_intermediaria",
        "amount": "valor_transfer",
        "isFraud": "fraude_transfer"
    })
)

cashouts_all = (
    df.loc[df["type"] == "CASH_OUT",
           ["step", "nameOrig", "nameDest", "amount", "isFraud"]]
    .rename(columns={
        "step": "step_cashout",
        "nameOrig": "origem_cashout",
        "nameDest": "destino_cashout",
        "amount": "valor_cashout",
        "isFraud": "fraude_cashout"
    })
)

# Procurar transferências cujo destino é a origem de um saque
candidatas_all = transfers_all.merge(
    cashouts_all,
    left_on="conta_intermediaria",
    right_on="origem_cashout",
    how="inner"
)

# Exigir que o saque ocorra em um step posterior
sequencias_all = candidatas_all.loc[
    candidatas_all["step_cashout"] > candidatas_all["step_transfer"]
].copy()

print("Pares conectados com saque em step posterior:",
      len(sequencias_all))

print("\nRótulos das transferências e saques:")
display(
    pd.crosstab(
        sequencias_all["fraude_transfer"],
        sequencias_all["fraude_cashout"],
        rownames=["TRANSFER isFraud"],
        colnames=["CASH_OUT isFraud"],
        margins=True
    )
)

print("\nExemplos de sequências encontradas:")
display(
    sequencias_all.sort_values(
        ["step_transfer", "step_cashout"]
    ).head(20)
)

Pares conectados com saque em step posterior: 265

Rótulos das transferências e saques:


CASH_OUT isFraud,0,All
TRANSFER isFraud,,
0,264,264
1,1,1
All,265,265



Exemplos de sequências encontradas:


,step_transfer,origem_transfer,conta_intermediaria,valor_transfer,fraude_transfer,step_cashout,origem_cashout,destino_cashout,valor_cashout,fraude_cashout
1,8,C1247903845,C874023329,924126.07,0,47,C874023329,C1468339976,292696.43,0
2,8,C1328840545,C874023329,94236.15,0,47,C874023329,C1468339976,292696.43,0
3,8,C1708840874,C874023329,374443.44,0,47,C874023329,C1468339976,292696.43,0
0,8,C1739297732,C2013691076,587279.16,0,132,C2013691076,C1228316980,433722.00,0
4,9,C159600904,C874023329,1861448.05,0,47,C874023329,C1468339976,292696.43,0
6,9,C374824020,C874023329,2600184.87,0,47,C874023329,C1468339976,292696.43,0
7,9,C830754054,C2013691076,214581.77,0,132,C2013691076,C1228316980,433722.00,0
5,9,C699554725,C2097141675,3022469.20,0,209,C2097141675,C1862512636,31292.34,0
11,10,C2109236358,C874023329,1137328.00,0,47,C874023329,C1468339976,292696.43,0
10,10,C2082616514,C2013691076,508308.16,0,132,C2013691076,C1228316980,433722.00,0


 Analisamos apenas os pares temporais já encontrados.

In [36]:
# Remover repetições de transferências causadas pelo merge
transferencias_unicas = sequencias_all.drop_duplicates(
    subset=[
        "step_transfer",
        "origem_transfer",
        "conta_intermediaria",
        "valor_transfer",
        "fraude_transfer"
    ]
)

# Remover repetições de saques causadas pelo merge
saques_unicos = sequencias_all.drop_duplicates(
    subset=[
        "step_cashout",
        "origem_cashout",
        "destino_cashout",
        "valor_cashout",
        "fraude_cashout"
    ]
)

# Resumo das transferências recebidas por conta intermediária
resumo_transferencias = (
    transferencias_unicas
    .groupby("conta_intermediaria")
    .agg(
        remetentes_distintos=("origem_transfer", "nunique"),
        qtd_transferencias=("origem_transfer", "size"),
        valor_total_recebido=("valor_transfer", "sum"),
        transferencias_rotuladas_fraude=("fraude_transfer", "sum"),
        primeiro_step_transfer=("step_transfer", "min"),
        ultimo_step_transfer=("step_transfer", "max")
    )
)

# Resumo dos saques posteriores por conta intermediária
resumo_saques = (
    saques_unicos
    .groupby("conta_intermediaria")
    .agg(
        qtd_saques_posteriores=("step_cashout", "size"),
        valor_total_sacado=("valor_cashout", "sum"),
        saques_rotulados_fraude=("fraude_cashout", "sum"),
        primeiro_step_saque=("step_cashout", "min")
    )
)

# Combinar os indicadores
ranking_contas = (
    resumo_transferencias
    .join(resumo_saques, how="inner")
    .reset_index()
    .sort_values(
        ["remetentes_distintos", "qtd_saques_posteriores"],
        ascending=False
    )
)

print("Contas intermediárias analisadas:", len(ranking_contas))

display(ranking_contas.head(20))

Contas intermediárias analisadas: 116


,conta_intermediaria,remetentes_distintos,qtd_transferencias,valor_total_recebido,transferencias_rotuladas_fraude,primeiro_step_transfer,ultimo_step_transfer,qtd_saques_posteriores,valor_total_sacado,saques_rotulados_fraude,primeiro_step_saque
109,C874023329,9,9,8565639.74,0,8,36,1,292696.43,0,47
66,C2088638273,6,6,3751366.50,0,12,41,1,103732.66,0,469
68,C2097141675,6,6,8911747.80,0,9,188,1,31292.34,0,209
85,C418053001,6,6,4790363.47,0,10,19,1,125278.40,0,153
30,C1598002267,5,5,2617583.31,0,43,279,1,379483.03,0,330
57,C1970551288,5,5,2485519.70,0,16,178,1,126007.67,0,357
60,C2013691076,5,5,3475945.11,0,8,129,1,433722.00,0,132
74,C239812349,5,5,6092764.65,0,22,286,1,124623.53,0,395
77,C279759786,5,5,3609067.51,0,139,349,1,90332.00,0,351
114,C986334426,5,5,5323004.27,0,12,47,1,104969.79,0,185


Reconstrução do histórico das contas intermediárias

In [37]:
# Selecionar as cinco contas mais bem classificadas
contas_alvo = ranking_contas.head(5)[
    "conta_intermediaria"
].tolist()

colunas = [
    "step", "type", "nameOrig", "nameDest",
    "amount", "isFraud", "isFlaggedFraud"
]

# Recuperar todas as transações que envolvem as contas selecionadas
historico = df.loc[
    df["nameOrig"].isin(contas_alvo)
    | df["nameDest"].isin(contas_alvo),
    colunas
].copy()

# Identificar o papel de cada conta-alvo na transação
historico["papel_conta_alvo"] = "OUTRO"

historico.loc[
    historico["nameOrig"].isin(contas_alvo),
    "papel_conta_alvo"
] = "ORIGEM"

historico.loc[
    historico["nameDest"].isin(contas_alvo),
    "papel_conta_alvo"
] = "DESTINO"

print("Contas selecionadas:", contas_alvo)
print("Transações no histórico completo:", len(historico))

print("\nResumo por conta, papel e tipo de transação:")

resumo_historico = (
    historico.assign(
        conta_alvo=historico["nameOrig"].where(
            historico["nameOrig"].isin(contas_alvo),
            historico["nameDest"]
        )
    )
    .groupby(["conta_alvo", "papel_conta_alvo", "type"])
    .agg(
        quantidade=("amount", "size"),
        valor_total=("amount", "sum"),
        transacoes_rotuladas_fraude=("isFraud", "sum")
    )
    .reset_index()
    .sort_values(
        ["conta_alvo", "papel_conta_alvo", "type"]
    )
)

display(resumo_historico)

print("\nHistórico cronológico da primeira conta:")

conta_foco = contas_alvo[0]

display(
    historico.loc[
        (historico["nameOrig"] == conta_foco)
        | (historico["nameDest"] == conta_foco)
    ]
    .sort_values(["step", "type"])
    .reset_index(drop=True)
)

Contas selecionadas: ['C874023329', 'C2088638273', 'C2097141675', 'C418053001', 'C1598002267']
Transações no histórico completo: 209

Resumo por conta, papel e tipo de transação:


,conta_alvo,papel_conta_alvo,type,quantidade,valor_total,transacoes_rotuladas_fraude
0,C1598002267,DESTINO,CASH_IN,3,528671.25,0
1,C1598002267,DESTINO,CASH_OUT,16,4102502.39,0
2,C1598002267,DESTINO,TRANSFER,5,2617583.31,0
3,C1598002267,ORIGEM,CASH_OUT,1,379483.03,0
4,C2088638273,DESTINO,CASH_IN,9,1979805.72,0
5,C2088638273,DESTINO,CASH_OUT,18,3722743.19,0
6,C2088638273,DESTINO,DEBIT,1,18356.12,0
7,C2088638273,DESTINO,TRANSFER,6,3751366.50,0
8,C2088638273,ORIGEM,CASH_OUT,1,103732.66,0
9,C2097141675,DESTINO,CASH_IN,13,2983293.97,0



Histórico cronológico da primeira conta:


,step,type,nameOrig,nameDest,amount,isFraud,isFlaggedFraud,papel_conta_alvo
0,8,CASH_IN,C1651517797,C874023329,30050.06,0,0,DESTINO
1,8,CASH_OUT,C1496902435,C874023329,305882.78,0,0,DESTINO
2,8,CASH_OUT,C982021784,C874023329,5675.26,0,0,DESTINO
3,8,CASH_OUT,C1328866120,C874023329,279753.71,0,0,DESTINO
4,8,CASH_OUT,C1926355203,C874023329,132760.22,0,0,DESTINO
5,8,CASH_OUT,C1875642026,C874023329,525333.34,0,0,DESTINO
6,8,CASH_OUT,C1953156554,C874023329,410974.71,0,0,DESTINO
7,8,CASH_OUT,C817905604,C874023329,415094.57,0,0,DESTINO
8,8,TRANSFER,C1247903845,C874023329,924126.07,0,0,DESTINO
9,8,TRANSFER,C1328840545,C874023329,94236.15,0,0,DESTINO


 Preparação do primeiro subgrafo

In [38]:
from pathlib import Path

In [39]:
# As cinco contas que queremos investigar
contas_alvo = ranking_contas.head(5)[
    "conta_intermediaria"
].tolist()

# Recuperar todas as transações que envolvem pelo menos
# uma das cinco contas, preservando o índice original
# como identificador único do evento neste dataset.
mascara = (
    df["nameOrig"].isin(contas_alvo)
    | df["nameDest"].isin(contas_alvo)
)

subgrafo = df.loc[mascara, [
    "step",
    "type",
    "nameOrig",
    "nameDest",
    "amount",
    "isFraud",
    "isFlaggedFraud"
]].copy()

subgrafo.insert(
    0,
    "tx_id",
    subgrafo.index.astype(str)
)

# Renomear para explicitar o papel de cada campo
subgrafo = subgrafo.rename(columns={
    "nameOrig": "source_account",
    "nameDest": "target_account"
})

# Exportar somente os campos necessários para o primeiro grafo
arquivo_saida = Path(
    "C:/Users/SAMSUNG/Desktop/"
    "deteccao de fraudes financeiras/"
    "NGIF03_subgrafo_contas.csv"
)

subgrafo.to_csv(
    arquivo_saida,
    index=False,
    encoding="utf-8"
)

print("Contas centrais:", len(contas_alvo))
print("Transações exportadas:", len(subgrafo))
print("IDs de transação únicos:", subgrafo["tx_id"].nunique())
print("Arquivo gerado:", arquivo_saida)

display(subgrafo.head(10))


Contas centrais: 5
Transações exportadas: 209
IDs de transação únicos: 209
Arquivo gerado: C:\Users\SAMSUNG\Desktop\deteccao de fraudes financeiras\NGIF03_subgrafo_contas.csv


,tx_id,step,type,source_account,target_account,amount,isFraud,isFlaggedFraud
16912,16912,8,CASH_OUT,C1496902435,C874023329,305882.78,0,0
18054,18054,8,CASH_OUT,C982021784,C874023329,5675.26,0,0
19795,19795,8,CASH_OUT,C1328866120,C874023329,279753.71,0,0
19918,19918,8,CASH_OUT,C1926355203,C874023329,132760.22,0,0
21177,21177,8,TRANSFER,C1247903845,C874023329,924126.07,0,0
22802,22802,8,CASH_OUT,C1875642026,C874023329,525333.34,0,0
25124,25124,8,CASH_IN,C1651517797,C874023329,30050.06,0,0
26241,26241,8,CASH_OUT,C1953156554,C874023329,410974.71,0,0
26768,26768,8,TRANSFER,C1328840545,C874023329,94236.15,0,0
26862,26862,8,TRANSFER,C1708840874,C874023329,374443.44,0,0


In [ ]:
# Alexandre Lotti
# <!LOTTI>